In [1]:
# import libraries
import json
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForTokenClassification, logging
import torch
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
import sys

# suppress unproblematic warnings from transformer models
logging.set_verbosity_error()

# load custom functions
sys.path.append("../../utils/")
from custom_evaluation import evaluate_seqeval, mention_level_evaluation, sentence_level_evaluation
from classification import tune_bert_ner, train_bert_ner

In [2]:
# load the annotated data in json format
with open("../../01_data/annotations_reduced.json", "r") as f:
    data = json.load(f)

# initialize tag dictionary
tag_dict = {"O"}

# loop through all sentences
for task in data:
    if task["annotations"]:
        for annotation in task["annotations"]:
            label = annotation["tag"][0:2]
            tag_dict.add(f"B-{label}")
            tag_dict.add(f"I-{label}")

# sort the tag dictionary
tag_list = sorted(tag_dict)

# dictionaries that convert from id to tag and vice versa
tag_to_id = {tag: i for i, tag in enumerate(tag_list)}
id_to_tag = {id: label for label, id in tag_to_id.items()}

In [3]:
# reduce the dataset for testing purposes
data = data[0:500]

In [4]:
# function that creates BIO-tags for text
def bert_tokenization_labelling(text, entities, tokenizer, tag2id, max_len):

    # tokenize and get offsets
    encoding = tokenizer(text, return_offsets_mapping=True, truncation=True, max_length=max_len, padding="max_length")
    
    # initialize label list with as many "O" labels as there are tokens
    tags = ["O"] * len(encoding.offset_mapping)
    
    # loop through all spans, get start and end position as well as the label
    for ent in entities:
        start, end = ent["start"], ent["end"]
        ent_tag = ent["tag"][0:2]
        
        # loop through all tokens in the sentence
        for idx, (token_start, token_end) in enumerate(encoding.offset_mapping):

            # if token is at the start of the annotation, this is the B token
            if token_start == start:
                tags[idx] = f"B-{ent_tag}"
            # if token starts after the start and before the end (end is character after the last) this is an I token
            if token_start > start and token_end <= end:
                tags[idx] = f"I-{ent_tag}"

    # convert tags to IDs, using -100 for padding/special tokens
    tag_ids = [-100 if word_id is None else tag2id.get(tag, tag2id["O"]) for tag, word_id in zip(tags, encoding.word_ids())]

    return encoding["input_ids"], encoding["attention_mask"], tag_ids, encoding.word_ids()

In [5]:
# split into training, validation and test dataset
train_val_data, test_data = train_test_split(data, test_size=0.2, random_state=42)
train_data, val_data = train_test_split(train_val_data, test_size=0.25, random_state=42)

class TokenDataset(Dataset):
    def __init__(self, data, tokenizer, tag2id, max_len=128):
        self.dataset = []
        self.max_len = max_len

        for task in data:
            # get the sentence and all annotations
            text = task["sentence"]
            spans = task["annotations"]

            # tokenize and get all ids
            input_ids, attention_mask, tag_ids, word_ids = bert_tokenization_labelling(text, spans, tokenizer, tag2id, max_len=self.max_len)

            # add everything to the dataset list
            self.dataset.append({
                "input_ids": torch.tensor(input_ids, dtype=torch.long),
                "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
                "tag_ids": torch.tensor(tag_ids, dtype=torch.long),
                "word_ids": word_ids})
  

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        return self.dataset[idx]

def custom_collate_fn(batch):
    input_ids = torch.stack([item["input_ids"] for item in batch])
    attention_masks = torch.stack([item["attention_mask"] for item in batch])
    tag_ids = torch.stack([item["tag_ids"] for item in batch])
    word_ids = [item["word_ids"] for item in batch]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_masks,
        "tag_ids": tag_ids,
        "word_ids": word_ids
    }

In [6]:
# define the search space to look into and the number of trials to take
search_space = {
    "lr": [9e-6, 2e-5, 4e-5],
    "batch_size": [8, 16, 32],
    "weight_decay": [0.01, 0.1, 0.3],
    "epochs": [2]
}
num_trials = 4

# load the tokenizer and model
model_names = ["roberta-base", "bert-base-cased", "distilbert-base-cased"] #, "microsoft/deberta-v3-base"

# set the device explicitly
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

optimal_configs = {}

for model_name in model_names:

    # define the respective tokenizer fitting to the model
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # create train and validation datasets
    train_dataset = TokenDataset(data=train_data, tokenizer=tokenizer, tag2id=tag_to_id)
    val_dataset = TokenDataset(data=val_data, tokenizer=tokenizer, tag2id=tag_to_id)

    # tune the bert model
    best_f1, optimal_params = tune_bert_ner(num_trials=num_trials, search_space=search_space, train_dataset=train_dataset, val_dataset=val_dataset,
                                   collate_fn=custom_collate_fn, model_name=model_name, tag2id=tag_to_id, id2tag=id_to_tag, device=device)
    
    # save the optimal configuration
    optimal_configs[model_name] = {
        "best_f1": best_f1,
        "best_params": optimal_params
    }


Starting search for model roberta-base
----------------------------------------------------------------------------------------------------

Trial 1 with params: {'lr': 2e-05, 'batch_size': 16, 'weight_decay': 0.01, 'epochs': 2}


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Epoch 1/2


Training: 100%|██████████| 19/19 [00:08<00:00,  2.33it/s, loss=0.159]


Average training loss: 0.3822


...

Average training loss: 0.1170
----------------------------------------------------------------------------------------------------

Trial 4 with params: {'lr': 9e-06, 'batch_size': 8, 'weight_decay': 0.01, 'epochs': 2}
Epoch 1/2


Training: 100%|██████████| 38/38 [00:04<00:00,  8.18it/s, loss=0.452]


Average training loss: 0.4711
Epoch 2/2


Training: 100%|██████████| 38/38 [00:04<00:00,  8.33it/s, loss=0.113] 


Average training loss: 0.1583
----------------------------------------------------------------------------------------------------
Best F1: 0.4000 with params: {'lr': 4e-05, 'batch_size': 16, 'weight_decay': 0.3, 'epochs': 2}


In [10]:
# train all models (again) with the best configurations

# empty dictionary to store the metrics on the test set
test_metrics = {}

# loop over the different models
for model_name in model_names:

    # get the optimal configuration for that model type
    batch_size = optimal_configs[model_name]["best_params"]["batch_size"]
    lr = optimal_configs[model_name]["best_params"]["lr"]
    weight_decay = optimal_configs[model_name]["best_params"]["weight_decay"]
    epochs = optimal_configs[model_name]["best_params"]["epochs"]

    # define the respective tokenizer fitting to the model
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # create datasets and dataloaders
    train_dataset = TokenDataset(data=train_data, tokenizer=tokenizer, tag2id=tag_to_id)
    test_dataset = TokenDataset(data=val_data, tokenizer=tokenizer, tag2id=tag_to_id)

    train_dataloader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=custom_collate_fn
    )
    test_dataloader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=custom_collate_fn
    )

    # set the device explicitly
    device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

    # create the model, optimizer
    model = AutoModelForTokenClassification.from_pretrained(model_name,
                                                            num_labels=len(tag_to_id),
                                                            id2label=id_to_tag,
                                                            label2id=tag_to_id).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    # train the model with the optimal configuration
    train_bert_ner(train_dataloader, model, optimizer, epochs, device)

    # apply all evaluation functions and save in the dictionary
    metrics_seqeval = evaluate_seqeval(model, test_dataloader, id_to_tag, device)
    metrics_cross_span = mention_level_evaluation(test_dataloader, model, device, id_to_tag)
    metrics_sentence_level = sentence_level_evaluation(test_dataloader, model, device, id_to_tag)

    # append all metrics to the dictionary
    test_metrics[model_name] = {
        "seqeval": metrics_seqeval,
        "cross_span": metrics_cross_span,
        "sentence_level": metrics_sentence_level
    }

Epoch 1/2


Training: 100%|██████████| 38/38 [00:09<00:00,  4.04it/s, loss=0.182] 


Average training loss: 0.2873
Epoch 2/2


Training: 100%|██████████| 38/38 [00:09<00:00,  4.20it/s, loss=0.0149] 


Average training loss: 0.0957
Epoch 1/2


...

Average training loss: 0.0846
Epoch 1/2


Training: 100%|██████████| 19/19 [00:03<00:00,  4.83it/s, loss=0.412]


Average training loss: 0.3734
Epoch 2/2


Training: 100%|██████████| 19/19 [00:03<00:00,  4.89it/s, loss=0.0376]


Average training loss: 0.1376


In [12]:
# export the test metrics
with open("evaluation_metrics_bert.json", "w") as f:
    json.dump(test_metrics, f, indent=4)

In [30]:
def predict_sentence(sentence, model, tokenizer, id_to_label, device='mps'):

    model.eval()

    # tokenize the input sentence
    encoding = tokenizer(
        sentence,
        return_tensors="pt",
        return_offsets_mapping=True,
        truncation=True,
        max_length=128
    )

    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)
    offset_mapping = encoding["offset_mapping"][0]

    # inference with autocast for mixed precision
    with torch.no_grad():
        with torch.autocast(device_type=device, dtype=torch.float16):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            predictions = torch.argmax(logits, dim=-1)[0]

    tokens = tokenizer.convert_ids_to_tokens(input_ids[0])
    labels = [id_to_label[pred.item()] for pred in predictions]

    result = []
    for token, label, (start, end) in zip(tokens, labels, offset_mapping):
        if start == 0 and end == 0:
            continue
        token_text = sentence[start:end]
        result.append((token_text, label))

    return result

In [31]:
# try out a custom sentence
text = "We want to protect young people."
predict_sentence(text, model=model, tokenizer=tokenizer, id_to_label=id_to_tag)

[('We', 'O'),
 (' want', 'O'),
 (' to', 'O'),
 (' protect', 'O'),
 (' young', 'O'),
 (' people', 'I-sg'),
 ('.', 'O')]